# METRICAS DE CLASIFICACIÓN. CURVAS PRECISION-RECALL. CURVAS ROC. AUC

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split


import warnings

warnings.filterwarnings('ignore')


%matplotlib inline



## Carga y vista previa del dataset

Para desarrollar este apartado vamos a utiizar el **dataset de scikit "breast cancer"**:

In [ ]:
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True, as_frame=True)

X.shape, y.shape

**Echamos un vistazo a los atributos:**

In [ ]:
X.head()

In [ ]:
X.columns

In [ ]:
# "arreglamos" nombres columnas atributos
columnas = [columna.replace(" ","_") for columna in X.columns]

X.columns = columnas

X.columns

In [ ]:
X.info()

Vemos que **todos los atributos son numéricos y que no contienen valores nulos**. En total son 30 atributos referentes a formas vistas en una radiografía.

**Examinamos ahora la variable objetivo (target):**

In [ ]:
y.unique()

In [ ]:
y.value_counts()

In [ ]:
# En tanto por ciento
y.value_counts(normalize=True)*100


De la documentación se desprende que **la codificación corresponde a :  (0) Maligno;    (1) Benigno.**

## EXPLORACIÓN DEL DATASET

**Creamos el dataframe, para inspeccionar más comodamente los datos:**

In [ ]:
bc_df = X.copy()

bc_df['diagnostico']= y

bc_df.info()

In [ ]:
# solo tiene atributos numéricos

target = 'diagnostico'

var_num = bc_df.columns.to_list()

var_num.remove(target)


### Distribución atributos

In [ ]:
# creamos grafica
fig, axes = plt.subplots(10, 3, figsize=(18, 20), gridspec_kw={'hspace': 1.5, 'wspace': 0.4})

ax = axes.ravel()

# esta paleta tiene 10 colores
colores = sns.color_palette("bright")


for idx,numerico in enumerate(var_num):
    sns.histplot(x=numerico, data=bc_df, kde=True, ax=ax[idx], bins=30, color=colores[idx%len(colores)])

    ax[idx].set_title(f'HISTOGRAMA {numerico}')




### Distribución variable objetivo (target)

In [ ]:
fig = plt.figure(figsize=(5,3))
ax1 = fig.add_subplot()

sns.countplot(x=target, data=bc_df)
ax1.set_title(f'DISTRIBUCION {target}')
ax1.set_xticklabels(['maligno (0)', 'benigno (1)'])


## ENTRENAMIENTO DEL MODELO. RESULTADOS

Vamos a utilizar **Naive bayes**. Como **modelo base** tomaremos **el clasificador KNN**

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier


**Partición TRAIN - TEST**

Como no hemos tenido necesidad de modificar los valores de los atributos y de la variable target, nos valen la "X" y la "y" devueltas por la función que cargó el dataset al inicio del notebook:

In [ ]:
print(f'{type(X)} \t"X" (atributos): {X.shape}')
print(f'{type(y)} \t"y" (target): {y.shape}')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train.shape, X_test.shape

### **Modelo base**

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)

Para medir su desempeño utilizaremos **dos métricas**: **'accuracy' por el acierto y 'precision' por los Falsos Positivos (recordad la codificación del target, "1" es BENIGNO):**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score


In [ ]:
y_pred = knn.predict(X_test)

accuracy_score(y_test, y_pred)

In [ ]:
precision_score(y_test, y_pred)

Son valores altos. Echemos un vistazo a la **matriz de confusión:**

In [ ]:
from sklearn.metrics import confusion_matrix

mat_conf = confusion_matrix(y_test, y_pred)

mat_conf



In [ ]:
# crear dataframe auxiliar
dataframe = pd.DataFrame(mat_conf, index=['maligno', 'benigno'], columns=['maligno', 'benigno'])

# crear heatmap
sns.heatmap(dataframe, annot=True, cbar=True, cmap='Reds')
plt.title('Confusion Matrix')

plt.tight_layout(), plt.xlabel('Predicted Values'), plt.ylabel('True Values');


### Modelo Naive Bayes

Entrenamos el modelo:

In [ ]:
nb = GaussianNB().fit(X_train,y_train)

In [ ]:
y_pred = nb.predict(X_test)

Veamos las métricas y la matriz de confusión:

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
precision_score(y_test, y_pred)

In [ ]:
mat_conf = confusion_matrix(y_test, y_pred)

# crear dataframe auxiliar
dataframe = pd.DataFrame(mat_conf, index=['maligno', 'benigno'], columns=['maligno', 'benigno'])

# crear heatmap
sns.heatmap(dataframe, annot=True, cbar=True, cmap='Reds')
plt.title('Confusion Matrix')

plt.tight_layout(), plt.xlabel('Predicted Values'), plt.ylabel('True Values')

Si quisieramos un **informe general con todas las métricas** utilizaremos **"classification_report"** (para binaria fijarse solo en la fila de la clase positiva):


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

### CURVAS PRECISION-RECALL

Las **curvas precision-recall** se obtienen **cambiando el umbral de decisión para las predicciones en forma de probabilidad. Éstas las obtenemos con el método "predict_proba" del modelo:**

In [ ]:
nb.predict_proba(X_test)[:5]

Nosotros seleccionaremos la de la clase positiva:

In [ ]:
nb.predict_proba(X_test)[:,1][:5]

In [ ]:
y_pred_proba = nb.predict_proba(X_test)[:,1]

**Pasamos a dibujar la curva con la función "precision_recall_curve" pasándole como parámetros "y_test" e "y_pred_proba":**

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

plt.fill_between(recall, precision)

plt.ylabel("Precision")
plt.xlabel("Recall")
plt.title("Train Precision-Recall curve");

Aparte de dibujar la curva, **los valores que devuelve la función se corresponden con las métricas "precicion" y "recall" para cada valor de umbral de probabilidad de la clase positiva ("thresholds"):**

In [ ]:
type(precision), type(recall), type(thresholds)

In [ ]:
precision.shape, recall.shape, thresholds.shape

In [ ]:
precision

In [ ]:
thresholds

### CURVAS ROC. Area bajo la curva AUC

Las **curvas ROC** se obtienen procediendo de forma similar, esta vez con la función **"roc_curve":**

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

plt.plot(fpr, tpr)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

Y podemos conocer el área bajo la curva con **"roc_auc_score"**, que **nos da una buena medida general de lo bueno que es nuestro clasificador:**

In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test, y_pred_proba)

Finalmente podemos **comparar los dos modelos, el base KNN y el utilizado NB**. Calculamos primero la predicción en probabilidad del KNN:

In [ ]:
y_pred_proba_knn = knn.predict_proba(X_test)[:,1]

fpr_knn, tpr_knn, threshold_knn = roc_curve(y_test, y_pred_proba_knn)

In [ ]:
# Dibujamos con todos los datos calculados

plt.plot(fpr, tpr, marker='^', label='Naive Bayes')
plt.plot(fpr_knn, tpr_knn, marker='.', label='KNN')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.legend()

In [ ]:
print(f'AUC (KNN): {roc_auc_score(y_test, y_pred_proba_knn)}')

print(f'AUC (Naive Bayes): {roc_auc_score(y_test, y_pred_proba)}')



### Mejorar resultados: desplazar el umbral de decisión.

**En algunos casos jugar con el umbral de decisión (moverlo de 0.5) nos puede hacer mejorar los resultados**. Empecemos por el punto de partida:

In [ ]:
# crear heatmap
sns.heatmap(dataframe, annot=True, cbar=True, cmap='Reds')
plt.title('Confusion Matrix')

plt.tight_layout(), plt.xlabel('Predicted Values'), plt.ylabel('True Values')

**Tenemos los valores de precision que queremos mejorar (menos Falsos Positivos) junto con sus valores de threshold, que calculamos cuando dibujamos la curva precision-recall:**

In [ ]:
precision

In [ ]:
thresholds

Comparando resultados vemos que **necesitamos de un umbral demasiado alto para mejorar los resultados que queremos**:

In [ ]:
y_pred2 = y_pred_proba >= 0.9999

y_pred2

In [ ]:
y_pred2.astype('int')

In [ ]:
mat_conf2 = confusion_matrix(y_test, y_pred2)

# crear dataframe auxiliar
dataframe2 = pd.DataFrame(mat_conf2, index=['maligno', 'benigno'], columns=['maligno', 'benigno'])

# crear heatmap
sns.heatmap(dataframe2, annot=True, cbar=True, cmap='Reds')
plt.title('Confusion Matrix')

plt.tight_layout(), plt.xlabel('Predicted Values'), plt.ylabel('True Values')

**Conclusiones: ¿cuanto ha empeorado accuracy? ¿que ha pasado con los Falsos Negativos?**